In [12]:
import sys
import sklearn
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import numpy as np
import os

df_2023 = pd.read_csv('g2f_2023_phenotypic_clean_data.csv', encoding='latin-1')
df_2022 = pd.read_csv('g2f_2022_phenotypic_clean_data.csv', encoding='latin-1')
df_2021 = pd.read_csv('g2f_2021_phenotypic_clean_data.csv', encoding='latin-1')
df_2020 = pd.read_csv('g2f_2020_phenotypic_clean_data.csv', encoding='latin-1')
df_2019 = pd.read_csv('g2f_2019_phenotypic_clean_data.csv', encoding='latin-1')
weather_2023 = pd.read_csv('G2F_WeatherData_2023.csv')
weather_2022 = pd.read_csv('G2F_WeatherData_2022.csv')
weather_2021 = pd.read_csv('G2F_WeatherData_2021.csv')
weather_2020 = pd.read_csv('G2F_WeatherData_2020.csv')
weather_2019 = pd.read_csv('G2F_WeatherData_2019.csv')
weather_2018 = pd.read_csv('G2F_WeatherData_2018.csv')
weather_2017 = pd.read_csv('G2F_WeatherData_2017.csv')
weather_2016 = pd.read_csv('G2F_WeatherData_2016.csv')
weather_2015 = pd.read_csv('G2F_WeatherData_2015.csv')
weather_2014 = pd.read_csv('G2F_WeatherData_2014.csv')

weather_dict = {}
weather_dict['2014'] = weather_2014
weather_dict['2015'] = weather_2015
weather_dict['2016'] = weather_2016
weather_dict['2017'] = weather_2017
weather_dict['2018'] = weather_2018
weather_dict['2019'] = weather_2019
weather_dict['2020'] = weather_2020
weather_dict['2021'] = weather_2021
weather_dict['2022'] = weather_2022
weather_dict['2023'] = weather_2023


/var/folders/1s/kv1k50_n0sl11vwn99v6zfc00000gn/T/ipykernel_67910/837065631.py:9: DtypeWarning: Columns (0: Plot, 1: Pass) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2023 = pd.read_csv('g2f_2023_phenotypic_clean_data.csv', encoding='latin-1')
/var/folders/1s/kv1k50_n0sl11vwn99v6zfc00000gn/T/ipykernel_67910/837065631.py:11: DtypeWarning: Columns (0: Plot) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2021 = pd.read_csv('g2f_2021_phenotypic_clean_data.csv', encoding='latin-1')
/var/folders/1s/kv1k50_n0sl11vwn99v6zfc00000gn/T/ipykernel_67910/837065631.py:12: DtypeWarning: Columns (0: Pass, 1: Filler) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2020 = pd.read_csv('g2f_2020_phenotypic_clean_data.csv', encoding='latin-1')
/var/folders/1s/kv1k50_n0sl11vwn99v6zfc00000gn/T/ipykernel_67910/837065631.py:13: DtypeWarning: Columns (0: Filler [enter 'filler' or blank], 1: Possible subs, 2: Confirme

In [13]:
required_phenotypes = [
    'Stand Count [# of plants]',
    'Plant Height [cm]',
    'Ear Height [cm]',
    'Anthesis [days]',
    'Silking [days]',
    'Grain Yield (bu/A)'
]

def clean_phenotypicData(df):
    df['experiment'] = df['Field-Location'].apply(lambda x: str(x)[2:] if pd.notna(x) and len(str(x)) >= 3 else None)
    df_h1 = df[df['experiment'] == 'H1'].copy()
    initial_count = len(df_h1)
    df_clean = df_h1.dropna(subset=required_phenotypes).copy()
    removed_count = initial_count - len(df_clean)
    return df_clean

cleaned_dfs = []
for year, df in [('2023', df_2023), ('2022', df_2022), 
                 ('2021', df_2021), ('2020', df_2020), ('2019', df_2019)]:
    cleaned = clean_phenotypicData(df)
    cleaned['year'] = year
    cleaned_dfs.append(cleaned)

full_phenotypic_df = pd.concat(cleaned_dfs, ignore_index=True)

In [14]:
dailyWeather_list = []

for year, df in weather_dict.items(): 
    df_copy = df.copy()
    
    df_copy['datetime'] = pd.to_datetime(
        df_copy['Year'].astype(str) + '-' + 
        df_copy['Month'].astype(str).str.zfill(2) + '-' + 
        df_copy['Day'].astype(str).str.zfill(2) + ' ' + 
        df_copy['Time'],
        format='%Y-%m-%d %H:%M',
        errors='coerce'
    )
    df_copy['date'] = df_copy['datetime'].dt.date
    df_copy['Field-Location'] = df_copy['Experiment'].str[:2] + 'H1'
    
    temp_col = [col for col in df_copy.columns if 'Temperature' in col][0]
    humid_col = [col for col in df_copy.columns if 'Relative_Humidity' in col][0]
    solar_col = [col for col in df_copy.columns if 'Solar' in col][0]
    rain_col = [col for col in df_copy.columns if 'Rainfall' in col][0]
    wind_col = [col for col in df_copy.columns if 'Wind Gust' in col][0]
    for col in [temp_col, humid_col, solar_col, rain_col, wind_col]:
        df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
    
    initial_rows = len(df_copy)
    df_copy = df_copy.dropna(subset=[temp_col, humid_col, solar_col, rain_col, wind_col])
    
    daily = df_copy.groupby(['Field-Location', 'date']).agg({
        temp_col: ['mean', 'max', 'min'],
        humid_col: 'mean',
        solar_col: ['sum', 'max'],
        rain_col: 'sum',
        wind_col: 'max'
    }).round(2)
    
    colsList = []

    for col in daily.columns.values:
        processed_col_name = '_'.join(col).strip()
        colsList.append(processed_col_name)
    daily.columns = colsList
    
    daily = daily.rename(columns={
        f'{temp_col}_mean': 'Temperature.C_mean',
        f'{temp_col}_max': 'Temperature.C_max',
        f'{temp_col}_min': 'Temperature.C_min',
        f'{humid_col}_mean': 'Relative_Humidity_perc_mean',
        f'{solar_col}_sum': 'Solar_Radiation_sum',
        f'{solar_col}_max': 'Solar_Radiation_max',
        f'{rain_col}_sum': 'Rainfall_mm_sum',
        f'{wind_col}_max': 'Wind_Gust_m_s_max'
    })
    
    daily = daily.reset_index()
    daily['year'] = year
    
    dailyWeather_list.append(daily)
weather_daily = pd.concat(dailyWeather_list, ignore_index=True)


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

vcf_file = "inbreds_G2F_2014-2023_437k.vcf"

full_phenotypic_df[['female_parent', 'male_parent']] = full_phenotypic_df['Pedigree'].str.split('/', expand=True)

female_parents_list = full_phenotypic_df['female_parent'].unique().tolist()
callset = allel.read_vcf(vcf_file, samples=female_parents_list, fields=['calldata/GT', 'variants/CHROM', 'variants/POS'])

genotypes = allel.GenotypeArray(callset['calldata/GT'])
snp_matrix = genotypes.to_n_alt().T

imputer = SimpleImputer(strategy='mean')
snp_matrix_imputed = imputer.fit_transform(snp_matrix)

selector = VarianceThreshold(threshold=0.01) 
snp_matrix_filtered = selector.fit_transform(snp_matrix_imputed)

pca = PCA(n_components=50)
snp_pca = pca.fit_transform(snp_matrix_filtered)

num_clusters = 15
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(snp_pca)

loaded_samples = callset['samples']
cluster_map = dict(zip(loaded_samples, clusters))
full_phenotypic_df['female_cluster'] = full_phenotypic_df['female_parent'].map(cluster_map)